# Foodborne example

Transcriptome Analysis of *Listeria monocytogenes* Exposed to Beef Fat Reveals Antimicrobial and Pathogenicity Attenuation Mechanisms

**doi**: https://doi.org/10.1128/AEM.03027-20

Ref: Chen YY, Kommadath A, Vahmani P, Visvalingam J, Dugan MER, Yang X. 2021. Transcriptome Analysis of Listeria monocytogenes Exposed to Beef Fat Reveals Antimicrobial and Pathogenicity Attenuation Mechanisms. Appl Environ Microbiol 87:e03027-20.
https://doi.org/10.1128/AEM.03027-20

- The example was done just for C18:2n-6

Obj: functional enrichment analysis

## Setup and inputs

Install `ResPathExplorer` with the `notebooks` extra and run this notebook from
`Examples/Foodborne bacteria/`, after `prepare_data.ipynb`. The original study
spreadsheet is an external input; see `README.md` in this directory. KEGG, CARD,
and VFDB access requires an internet connection on the first run. Cached GMT,
ARO, and VFDB files are reused. These are annotations by gene name, not sequence
or phenotype validation. Saved outputs were cleared after the workflow fixes;
rerun with your study data to generate results.


In [ ]:
from pathlib import Path

required_inputs = [
    "upregulated.txt", "background.txt", "upregulatedGNames.txt",
    "downregulted.txt", "downregultedGNames.txt", "df_up.xlsx", "df_down.xlsx"
]
missing = [name for name in required_inputs if not Path(name).is_file()]
if missing:
    raise FileNotFoundError(
        "Run prepare_data.ipynb in this directory first. Missing: " + ", ".join(missing)
    )


## Upregulated genes

### KeggAnalysis

In [ ]:
from ResPathExplorer.KeggAnalysis import KeggAnalysis
organism = "Listeria monocytogenes EGD-e (serotype 1/2a)" #organism name
GMT_file = "exemple_fb.gmt" #GMT file name
example_01 = KeggAnalysis(organism_name = organism, file_name_gmt= GMT_file, use_existing_gmt = Path(GMT_file).is_file())

In [ ]:
print(example_01.org)
print(example_01.organism)

In [ ]:
with open('upregulated.txt', 'r') as file:
    upregulated = [line.strip() for line in file if line.strip()]
len(upregulated)

In [ ]:
with open('background.txt', 'r') as file:
    background = [line.strip() for line in file if line.strip()]
len(background)

In [ ]:
example_01.enrichment_analysis(gene_list = upregulated,
            cutoff = 0.05,
            name_outdir = "enrichment_exampleup",
            number_path = 10,
            name_results_file = "enrichment_exampleup",
            genes_background = background)

In [ ]:
uprregulatedpaths = example_01.enrichment_results
uprregulatedpaths

In [ ]:
example_01.visualize_enrichment_results(name_outdir = "enrichment_exampleup",
            outplot_file_name = "enrichment_exampleup",
            plot_title = "Gene set enrichment analysis",
            plot_type = "barplot")

### CARDAnalysis

In [ ]:
with open('upregulatedGNames.txt', 'r') as file:
    genes = [line.strip() for line in file if line.strip()]
len(genes)

In [ ]:
from ResPathExplorer.CARDAnalysis import CARDAnalysis
example_01AG = CARDAnalysis(genes_list = genes, has_CARDdata = Path("aro.obo").is_file())

In [ ]:
df_argup = example_01AG.ARGdf
df_argup

In [ ]:
from ResPathExplorer.save_df_as_html import save_df_as_html
save_df_as_html(df_argup, filename = "ARGs_res.html")

In [ ]:
if not df_argup.empty:
    example_01AG.plot_antibiotic_frequencies(df_argup)
else:
    print("No CARD gene-name matches in the upregulated group.")

### VFDBAnalysis

In [ ]:
from ResPathExplorer.VFDBAnalysis import VFDBAnalysis
example_01VF = VFDBAnalysis(db_dir="vfdb_data")

In [ ]:
df_genes = example_01VF.load_and_process()

In [ ]:
with open('upregulatedGNames.txt', 'r') as file:
    genes = [line.strip() for line in file if line.strip()]
    
bacteria_name = "Listeria monocytogenes"

df_vfup = example_01VF.search_virulence_genes(genes, bacteria_name)
df_vfup

In [ ]:
from ResPathExplorer.save_df_as_html import save_df_as_html
save_df_as_html(df_vfup, filename = "VFs_genes.html")

In [ ]:
# example_01VF.plot_virulence_factors_percentage(df_vf,
#                                           bacteria_name = bacteria_name,
#                                           show_all_categories = False)

### URL pathways

In [ ]:
ARGs = [x for x in example_01AG.ARGdf["Matched Name"]]
VFGs = [x for x in df_vfup["Gene_Name"]]
print(f"Genes names - general: {len(genes)}")
print(f"ARGs: {len(ARGs)}")
print(f"VFs genes: {len(VFGs)}")

In [ ]:
import pandas as pd
df_up = pd.read_excel("df_up.xlsx")

df_up["GName"] = df_up["GName"].fillna("").str.strip()
df_up["KEGG_ID"] = df_up["KEGG_ID"].str.split(":").str[-1]
vf_upgenes = set(df_vfup["Gene_Name"].dropna().str.strip())
arg_upgenes = set(df_argup["Matched Name"].dropna().str.strip())
vf_up_lookup = {gene.casefold() for gene in vf_upgenes}
arg_up_lookup = {gene.casefold() for gene in arg_upgenes}


def get_up_color(gene):
    gene = gene.casefold()
    if gene in vf_up_lookup:
        return "skyblue,purple"
    if gene in arg_up_lookup:
        return "indianred,blue"
    return "yellow,deeppink"


df_up["Color"] = df_up["GName"].apply(get_up_color)
gene_color_dict_up = df_up.set_index("KEGG_ID")["Color"].to_dict()


In [ ]:
vf_upgenes

In [ ]:
df_up.head()

In [ ]:
print(len(df_up[df_up["Color"] == "indianred,blue"]))
print(len(df_up[df_up["Color"] == "skyblue,purple"]))

In [ ]:
gene_color_dict_up

In [ ]:
term_list = []
for t in example_01.enrichment_results["Term"]:
    term_list.append(t)
term_list

In [ ]:
from ResPathExplorer.URL_pathway import get_url_pathway
gene_set = example_01.gene_set

for t in term_list:
    genes_in_pathway = next(genes for (path_id, _), genes in gene_set.items() if path_id == t)

    filtered_gene_color_dict = {
        gene: gene_color_dict_up[gene]
        for gene in genes_in_pathway
        if gene in gene_color_dict_up
    }
    url = get_url_pathway(t, filtered_gene_color_dict)
    print(f"URL {t}:", url)

## Downregulated genes

In [ ]:
from ResPathExplorer.KeggAnalysis import KeggAnalysis
organism = "Listeria monocytogenes EGD-e (serotype 1/2a)" #organism name
GMT_file = "exemple_fb.gmt" #GMT file name
example_02 = KeggAnalysis(organism_name = organism, file_name_gmt= GMT_file, use_existing_gmt = Path(GMT_file).is_file())

In [ ]:
with open('downregulted.txt', 'r') as file:
    downregulated = [line.strip() for line in file if line.strip()]
print(len(downregulated))
print(len(background))

In [ ]:
example_02.enrichment_analysis(gene_list = downregulated,
            cutoff = 0.05,
            name_outdir = "enrichment_exampledown",
            number_path = 10,
            name_results_file = "enrichment_exampledown",
            genes_background = background)

In [ ]:
downregulatedpaths = example_02.enrichment_results
downregulatedpaths

In [ ]:
example_02.visualize_enrichment_results(name_outdir = "enrichment_exampledown",
            outplot_file_name = "enrichment_exampledown",
            plot_title = "Gene set enrichment analysis",
            plot_type = "barplot")

### CARDAnalysis

In [ ]:
with open('downregultedGNames.txt', 'r') as file:
    genes = [line.strip() for line in file if line.strip()]
len(genes)

In [ ]:
from ResPathExplorer.CARDAnalysis import CARDAnalysis
example_02AG = CARDAnalysis(genes_list = genes, has_CARDdata = Path("aro.obo").is_file())

In [ ]:
df_argdown = example_02AG.ARGdf
df_argdown

### VFDBAnalysis

In [ ]:
from ResPathExplorer.VFDBAnalysis import VFDBAnalysis
example_02VF = VFDBAnalysis(db_dir="vfdb_data")

In [ ]:
df_genes = example_02VF.load_and_process()

In [ ]:
bacteria_name = "Listeria monocytogenes"

df_vfdown = example_02VF.search_virulence_genes(genes, bacteria_name)
df_vfdown

In [ ]:
if not df_vfdown.empty:
    example_02VF.plot_virulence_factors_percentage(
        df_vfdown, bacteria_name=bacteria_name, show_all_categories=False
    )
else:
    print("No VFDB gene-name matches in the downregulated group.")

### URL pathways

In [ ]:
ARGs = [x for x in example_02AG.ARGdf["Matched Name"]]
VFGs = [x for x in df_vfdown["Gene_Name"]]
print(f"Genes names - general: {len(genes)}")
print(f"ARGs: {len(ARGs)}")
print(f"VFs genes: {len(VFGs)}")

In [ ]:
import pandas as pd
df_down = pd.read_excel("df_down.xlsx")

df_down["GName"] = df_down["GName"].fillna("").str.strip()
df_down["KEGG_ID"] = df_down["KEGG_ID"].str.split(":").str[-1]
vf_downgenes = set(df_vfdown["Gene_Name"].dropna().str.strip())
arg_downgenes = set(df_argdown["Matched Name"].dropna().str.strip())
vf_down_lookup = {gene.casefold() for gene in vf_downgenes}
arg_down_lookup = {gene.casefold() for gene in arg_downgenes}


def get_down_color(gene):
    gene = gene.casefold()
    if gene in vf_down_lookup:
        return "skyblue,purple"
    if gene in arg_down_lookup:
        return "indianred,blue"
    return "yellow,deeppink"


df_down["Color"] = df_down["GName"].apply(get_down_color)
gene_color_dict_down = df_down.set_index("KEGG_ID")["Color"].to_dict()


In [ ]:
print(len(df_down[df_down["Color"] == "indianred,blue"]))
print(len(df_down[df_down["Color"] == "skyblue,purple"]))

In [ ]:
term_list = []
for t in example_02.enrichment_results["Term"]:
    term_list.append(t)
term_list

In [ ]:
from ResPathExplorer.URL_pathway import get_url_pathway
gene_set = example_02.gene_set

for t in term_list:
    genes_in_pathway = next(genes for (path_id, _), genes in gene_set.items() if path_id == t)

    filtered_gene_color_dict = {
        gene: gene_color_dict_down[gene]
        for gene in genes_in_pathway
        if gene in gene_color_dict_down
    }
    url = get_url_pathway(t, filtered_gene_color_dict)
    print(f"URL {t}:", url)

### Profile

In [ ]:
uprregulatedpathsnames = set(uprregulatedpaths["Pathway name"])
downregulatedpathsnames = set(downregulatedpaths["Pathway name"])

In [ ]:
import json

profile_dict = {
    "upregulated_genes": {
        "enriched_pathways": sorted(uprregulatedpathsnames),
        "resistance_genes": sorted(arg_upgenes),
        "virulence_genes": sorted(vf_upgenes)
    },
    "downregulated_genes": {
        "enriched_pathways": sorted(downregulatedpathsnames),
        "resistance_genes": sorted(arg_downgenes),
        "virulence_genes": sorted(vf_downgenes)
    }
}
print(json.dumps(profile_dict, indent=2))


In [ ]:
profile_dict